In [2]:
# 如果需要，可取消注释并改成你的路径
import sys
# sys.path.append(r'.')

from importlib import reload
import os
import low_cut_utils_modified as ref 
reload(ref)
from low_cut_utils_modified import *
import numpy as np
from lgdo import lh5
import numpy as np
import awkward as ak
from lgdo import lh5
import pandas as pd
import matplotlib.pyplot as plt


In [3]:
filepath = "/mnt/atlas01/users/vogl/scarf/sp01/data/hit_vcs"
files = os.listdir(filepath)
files.sort()
files

pnum_cal = "05"
run_cal = "057"

# bg run 6 days
pnum_phy_bg = "00"
run_phy_bg = "038"
# signal run 13 days
pnum_phy_signal = "05"
run_phy_signal = "059"

runs = [
    's100ns_mw1', 's100ns_mw3', 's100ns_mw5', 's100ns_mw7',
    's200ns_mw1', 's200ns_mw3', 's200ns_mw5', 's200ns_mw7',
    's60ns_mw1',  's60ns_mw3',  's60ns_mw5',  's60ns_mw7'
]



def build_run_db(runs, filepath):
    db = {}
    missing = []

    for run in runs:
        m = re.match(r"s(\d+)ns_mw(\d+)", run)
        if not m:
            missing.append(f"{run}: invalid run name format")
            continue

        run_path = os.path.join(filepath, run)

        phy_bg_path = os.path.join(run_path, "phy", f"p{pnum_phy_bg}", f"r{run_phy_bg}")
        phy_signal_path = os.path.join(run_path, "phy", f"p{pnum_phy_signal}", f"r{run_phy_signal}")
        cal_path = os.path.join(run_path, "cal", f"p{pnum_cal}", f"r{run_cal}")
        missing_this_run = []
        if not os.path.exists(run_path):
            missing_this_run.append(f"run folder missing: {run_path}")
        if not os.path.exists(phy_bg_path):
            missing_this_run.append(f"phy_bg path missing: {phy_bg_path}")
        if not os.path.exists(phy_signal_path):
            missing_this_run.append(f"phy_signal path missing: {phy_signal_path}")
        if not os.path.exists(cal_path):
            missing_this_run.append(f"cal path missing: {cal_path}")
        if missing_this_run:
            missing.append(f"{run}\n  " + "\n  ".join(missing_this_run))
            continue

        phy_bg_files = sorted(os.listdir(phy_bg_path))
        phy_signal_files = sorted(os.listdir(phy_signal_path))
        cal_files = sorted(os.listdir(cal_path))

        db[run] = {
            "run": run,
            "time_ns": int(m.group(1)),
            "mw": int(m.group(2)),
            "interval": f"{m.group(1)}ns",
            "mw_label": f"mw{m.group(2)}",
            "phy_bg_path": phy_bg_path,
            "phy_signal_path": phy_signal_path,
            "cal_path": cal_path,
            "phy_bg_files": phy_bg_files,
            "phy_signal_files": phy_signal_files,
            "cal_files": cal_files,
            "phy_bg_pathlist": [os.path.join(phy_bg_path, f) for f in phy_bg_files],
            "phy_signal_pathlist": [os.path.join(phy_signal_path, f) for f in phy_signal_files],
            "cal_pathlist": [os.path.join(cal_path, f) for f in cal_files],
        }

    if missing:
        raise FileNotFoundError(
            "[ERROR] Some runs or paths are missing:\n\n" + "\n\n".join(missing)
        )
    return db
run_db = build_run_db(runs, filepath)


In [ ]:
run = 's100ns_mw5'
low_cut_bool = safe_lh5_read(lh5,"ch002/hit",run_db[run]['cal_pathlist'])['A_max_mw_o_E_fix_Low_Cut']
aoe_field = safe_lh5_read(lh5,'ch002/hit',run_db[run]['cal_pathlist'])['A_max_mw_o_E_fix_Classifier']
energy_field = safe_lh5_read(lh5,'ch002/hit',run_db[run]['cal_pathlist'])['A_max_mw_o_E_fix_Classifier']

In [13]:
cut_value = aoe_field[np.where(low_cut_bool)==True]

In [21]:
cut_value

array([], shape=(0, 3090061), dtype=float64)

In [30]:
import numpy as np

# 1. 确保数据是 numpy 格式 (如果 safe_lh5_read 返回的是列表或 Panda Series)
aoe_values = np.array(aoe_field)
is_accepted = np.array(low_cut_bool)

# 2. 获取所有通过 Cut 的 A/E 值
accepted_aoe = aoe_values[is_accepted]

# 3. 找到最小值，这就是该运行 (run) 的 Cut 阈值
if len(accepted_aoe) > 0:
    cut_value = np.min(accepted_aoe)
    print(f"Run {run} 的 A/E Cut 阈值（最小值）为: {cut_value:.6f}")
else:
    print("没有发现通过 Cut 的事件，请检查数据。")

Run s100ns_mw5 的 A/E Cut 阈值（最小值）为: -1.114995


In [1]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. 数据准备与安全过滤 ---
# 确保 aoe_field 和 low_cut_bool 是 numpy 数组
aoe_vals = np.array(aoe_field)
passed_mask = np.array(low_cut_bool)

# 仅提取有效且通过 Cut 的数据
# 使用 & 操作符合并布尔掩码
valid_accepted = aoe_vals[np.isfinite(aoe_vals) & passed_mask]

# 计算 Cut 值（用于画垂线）
extracted_cut = np.min(valid_accepted) if len(valid_accepted) > 0 else 0

# --- 2. 生成直方图数据 ---
# 根据数据的分布动态设置范围（例如取 1% 到 99% 的分位数）
plot_range = (np.percentile(valid_accepted, 1), np.percentile(valid_accepted, 99))
counts, edges = np.histogram(valid_accepted, bins=100, range=plot_range)
centers = 0.5 * (edges[:-1] + edges[1:])

# --- 3. 绘图 ---
plt.figure(figsize=(10, 6), facecolor='white')

# 使用阶梯图绘制通过的事件
# where="mid" 确保阶梯的平台中心对齐 Bin 的中心点
plt.step(centers, counts, where="mid", color='blue', linewidth=1.5, label='Accepted Events (A/E > Cut)')

# 填充背景（可选，增加可读性）
plt.fill_between(centers, counts, step="mid", color='blue', alpha=0.1)

# 绘制 Cut 值的垂直线
plt.axvline(extracted_cut, color='red', linestyle='--', linewidth=2, 
            label=f'Low Cut Value: {extracted_cut:.5f}')

# --- 4. 格式化图表 ---
plt.yscale('log')  # A/E 分布通常在对数坐标下更清晰
plt.xlabel('A/E Classifier Value', fontsize=12)
plt.ylabel('Counts', fontsize=12)
plt.title(f'A/E Distribution - Run: {run}', fontsize=14)
plt.grid(True, which="both", alpha=0.2, linestyle='--')
plt.legend(loc='upper left')

plt.tight_layout()
plt.show()


KeyboardInterrupt



In [42]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Load and Define Parameters ---
# Assuming energy_array is already loaded from your lh5 read
energy_vals = np.array(energy_field)
aoe_vals = np.array(aoe_field)
passed_mask = np.array(low_cut_bool)

# Define DEP range (Double Escape Peak)
fitrange_dep = (1565, 1615) 

# --- 2. Create the DEP Range Mask ---
# We check for energy limits AND finite values AND valid waveforms
dep_range_mask = (energy_vals > fitrange_dep[0]) & (energy_vals < fitrange_dep[1])
finite_mask = np.isfinite(aoe_vals) & np.isfinite(energy_vals)

# --- 3. Combine Masks for the Plot ---
# Mask for ALL events in the DEP energy region
dep_total_mask = dep_range_mask & finite_mask

# Mask for events in the DEP region that PASSED the A/E cut
dep_accepted_mask = dep_total_mask & passed_mask

# --- 4. Extract Data for Plotting ---
total_dep_aoe = aoe_vals[dep_total_mask]
accepted_dep_aoe = aoe_vals[dep_accepted_mask]

# Calculate the cut value from the accepted events
extracted_cut = np.min(accepted_dep_aoe) if len(accepted_dep_aoe) > 0 else 0

# --- 5. Generate Step Plot ---
# We'll use a fixed range around 1.0 since A/E is usually normalized
plot_bins = 100
plot_range = (0.5, 1.5) # Adjust this based on your A/E distribution

counts_total, edges = np.histogram(total_dep_aoe, bins=plot_bins, range=plot_range)
counts_acc, _ = np.histogram(accepted_dep_aoe, bins=plot_bins, range=plot_range)
centers = 0.5 * (edges[:-1] + edges[1:])

plt.figure(figsize=(10, 6))

# Plot Total DEP events in Gray (Background + Signal)
plt.step(centers, counts_total, where="mid", color='gray', alpha=0.5, label='Total DEP Region Events')

# Plot Accepted DEP events in Blue (The SSE signal we keep)
plt.step(centers, counts_acc, where="mid", color='blue', linewidth=1.5, label='Accepted DEP (SSE)')

# Vertical line for the cut
plt.axvline(extracted_cut, color='red', linestyle='--', label=f'A/E Cut: {extracted_cut:.5f}')

plt.yscale('log')
plt.xlabel('A/E Classifier Value')
plt.ylabel('Counts')
plt.title(f'A/E Distribution at DEP (1592.5 keV) - Run: {run}')
plt.grid(True, which="both", alpha=0.2)
plt.legend()
plt.show()

NameError: name 'energy_field' is not defined